# Batch Expansion & Azure Classification

This notebook reads all `classified_index` + `stored_results` data from the source DB,
expands every matched snippet to its full surrounding clause **and** classifies it in a
single LLM call (the LLM decides clause boundaries instead of regex), then writes
everything into a new database named `icat-v2-beta`.

Supports checkpoint/resume so the notebook can be stopped and restarted without re-processing.

In [24]:
# Cell 1: Imports & Configuration
import sys
import os
import json
import time
from datetime import datetime

import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

import insert_data
import pipelineoperation

CHECKPOINT_FILE = "checkpoint_v2_beta.json"
BATCH_SIZE = 50
TARGET_DB_NAME = "icat-v2-beta"

print("Configuration loaded.")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  TARGET_DB_NAME = {TARGET_DB_NAME}")
print(f"  CHECKPOINT_FILE = {CHECKPOINT_FILE}")

Configuration loaded.
  BATCH_SIZE = 50
  TARGET_DB_NAME = icat-v2-beta
  CHECKPOINT_FILE = checkpoint_v2_beta.json


In [25]:
# Cell 2: Database Connections

# Source connection (uses env vars as-is)
source_conn = insert_data.create_connection()
assert source_conn is not None, "Failed to connect to source database"
print(f"Connected to source DB: {os.environ.get('DB_NAME')}")

# Create target database if it doesn't exist
admin_uri = (
    f"host={insert_data.DB_HOST} dbname={insert_data.DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
admin_conn = psycopg.connect(admin_uri, autocommit=True)
with admin_conn.cursor() as cur:
    try:
        cur.execute(f'CREATE DATABASE "{TARGET_DB_NAME}"')
        print(f"Created database '{TARGET_DB_NAME}'")
    except psycopg.errors.DuplicateDatabase:
        print(f"Database '{TARGET_DB_NAME}' already exists")
admin_conn.close()

# Connect to target database
target_uri = (
    f"host={insert_data.DB_HOST} dbname={TARGET_DB_NAME} "
    f"user={insert_data.DB_USER} password={insert_data.DB_PASS} "
    f"sslmode={insert_data.DB_SSLMODE}"
)
target_conn = psycopg.connect(target_uri)
print(f"Connected to target DB: {TARGET_DB_NAME}")

# Initialize schema in target
insert_data.initialize_db(target_conn)
print("Target DB schema initialized.")

Connected to source DB: icatv1
Database 'icat-v2-beta' already exists
Connected to target DB: icat-v2-beta
Database initialized
Target DB schema initialized.


In [ ]:
# Cell 3: Test Run — Sample 20 Docs Across 5 Queries
# Uses the combined expand_and_classify_with_azure pipeline:
# one LLM call per snippet that determines clause boundaries AND classifies.

import importlib
importlib.reload(pipelineoperation)
from pipelineoperation import expand_and_classify_with_azure

# Pick 5 queries that have non-empty matching data
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT searchquery FROM classified_index
        WHERE LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2
        LIMIT 5
    """)
    test_queries = [r[0] for r in cur.fetchall()]

print(f"Test queries: {test_queries}")

# Fetch 4 rows per query
test_rows = []
with source_conn.cursor() as cur:
    for q in test_queries:
        cur.execute("""
            SELECT doc_id, title, searchquery, matching_indents, matching_columns
            FROM classified_index WHERE searchquery = %s
            AND (LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2)
            LIMIT 4
        """, (q,))
        test_rows.extend(cur.fetchall())

print(f"Fetched {len(test_rows)} test rows")

# Prefetch Doc_Text for these docs
test_doc_ids = list({r[0] for r in test_rows})
test_doc_texts = {}
with source_conn.cursor() as cur:
    for doc_id in test_doc_ids:
        cur.execute("SELECT Doc_Text FROM stored_results WHERE Doc_ID = %s", (doc_id,))
        row = cur.fetchone()
        if row and row[0]:
            test_doc_texts[doc_id] = row[0]

print(f"Prefetched doc text for {len(test_doc_texts)} docs")

# Process each row: combined expand + classify in one LLM call
results_table = []
for row in test_rows:
    doc_id, title, query, mi_raw, mc_raw = row
    doc_text = test_doc_texts.get(doc_id, '')
    mc = insert_data._safe_eval(mc_raw)
    mi = insert_data._safe_eval(mi_raw)

    # Each call returns {"clause_text": ..., "is_contract_clause": bool}
    mc_results = [expand_and_classify_with_azure(doc_text, s) for s in mc]
    mi_results = [expand_and_classify_with_azure(doc_text, s) for s in mi]

    expanded_mc = [r['clause_text'] for r in mc_results]
    expanded_mi = [r['clause_text'] for r in mi_results]
    classified_mc = [r['clause_text'] for r in mc_results if r['is_contract_clause']]
    classified_mi = [r['clause_text'] for r in mi_results if r['is_contract_clause']]

    sample = (classified_mc[0][:200] if classified_mc else
              classified_mi[0][:200] if classified_mi else
              expanded_mc[0][:200] if expanded_mc else
              expanded_mi[0][:200] if expanded_mi else '\u2014')

    results_table.append({
        'Query': query,
        'DocID': doc_id,
        'Title': (title or '')[:50],
        'Snippets (mc)': len(mc),
        'Snippets (mi)': len(mi),
        'Expanded (mc)': len(expanded_mc),
        'Expanded (mi)': len(expanded_mi),
        'Classified (mc)': len(classified_mc),
        'Classified (mi)': len(classified_mi),
        'Sample Clause': sample,
        'Is Contract': ('Yes' if classified_mc or classified_mi else 'No'),
    })

df = pd.DataFrame(results_table)
display(df)
print("\n** Review the table above. Do not proceed to full processing until satisfied. **")

In [ ]:
df.to_csv("batch1_1.csv")

In [5]:
# Cell 3: Test Run — Sample 20 Docs Across 5 Queries
# Uses the combined expand_and_classify_with_azure pipeline:
# one LLM call per snippet that determines clause boundaries AND classifies.

from pipelineoperation import expand_and_classify_with_azure

# Pick 5 queries that have non-empty matching data
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT DISTINCT searchquery FROM classified_index
        WHERE LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2
        LIMIT 5
    """)
    test_queries = [r[0] for r in cur.fetchall()]

print(f"Test queries: {test_queries}")

# Fetch 4 rows per query
test_rows = []
with source_conn.cursor() as cur:
    for q in test_queries:
        cur.execute("""
            SELECT doc_id, title, searchquery, matching_indents, matching_columns
            FROM classified_index WHERE searchquery = %s
            AND (LENGTH(matching_columns) > 2 OR LENGTH(matching_indents) > 2)
            LIMIT 4
        """, (q,))
        test_rows.extend(cur.fetchall())

print(f"Fetched {len(test_rows)} test rows")

# Prefetch Doc_Text for these docs
test_doc_ids = list({r[0] for r in test_rows})
test_doc_texts = {}
with source_conn.cursor() as cur:
    for doc_id in test_doc_ids:
        cur.execute("SELECT Doc_Text FROM stored_results WHERE Doc_ID = %s", (doc_id,))
        row = cur.fetchone()
        if row and row[0]:
            test_doc_texts[doc_id] = row[0]

print(f"Prefetched doc text for {len(test_doc_texts)} docs")

# Process each row: combined expand + classify in one LLM call
results_table = []
for row in test_rows:
    doc_id, title, query, mi_raw, mc_raw = row
    doc_text = test_doc_texts.get(doc_id, '')
    mc = insert_data._safe_eval(mc_raw)
    mi = insert_data._safe_eval(mi_raw)

    # Each call returns {"clause_text": ..., "is_contract_clause": bool}
    mc_results = [expand_and_classify_with_azure(doc_text, s) for s in mc]
    mi_results = [expand_and_classify_with_azure(doc_text, s) for s in mi]

    expanded_mc = [r['clause_text'] for r in mc_results]
    expanded_mi = [r['clause_text'] for r in mi_results]
    classified_mc = [r['clause_text'] for r in mc_results if r['is_contract_clause']]
    classified_mi = [r['clause_text'] for r in mi_results if r['is_contract_clause']]

    sample = (classified_mc[0][:2000] if classified_mc else
              classified_mi[0][:2000] if classified_mi else
              expanded_mc[0][:2000] if expanded_mc else
              expanded_mi[0][:2000] if expanded_mi else '\u2014')

    results_table.append({
        'Query': query,
        'DocID': doc_id,
        'Title': (title or '')[:50],
        'Snippets (mc)': len(mc),
        'Snippets (mi)': len(mi),
        'Expanded (mc)': len(expanded_mc),
        'Expanded (mi)': len(expanded_mi),
        'Classified (mc)': len(classified_mc),
        'Classified (mi)': len(classified_mi),
        'Sample Clause': sample,
        'Is Contract': ('Yes' if classified_mc or classified_mi else 'No'),
    })

df = pd.DataFrame(results_table)
display(df)
print("\n** Review the table above. Do not proceed to full processing until satisfied. **")

Test queries: ['Arbitration ', 'Privacy', 'force majeure', 'confidentiality', 'contributor license agreement']
Fetched 17 test rows
Prefetched doc text for 15 docs


,Query,DocID,Title,Snippets (mc),Snippets (mi),Expanded (mc),Expanded (mi),Classified (mc),Classified (mi),Sample Clause,Is Contract
0,Arbitration,133653680,M/S Duro Felguera S.A vs M/S. Gangavaram Port ...,13,0,13,0,9,0,Any dispute in respect of which amicable settl...,Yes
1,Arbitration,81257869,Gurinder Singh Sethi vs M/S Sagar Fossil Fuel ...,1,0,1,0,1,0,27. That a mutually agreed arbitrator shall be...,Yes
2,Arbitration,9193128,Pure Diets India Limited vs Lokmangal Agro Ind...,2,0,2,0,0,0,(a). Nilesh C. Sanghani and Others v. Rakesh V...,No
3,Arbitration,133653680,M/S Duro Felguera S.A vs M/S. Gangavaram Port ...,13,0,13,0,9,0,Any dispute in respect of which amicable settl...,Yes
4,Privacy,12972852,My Space Inc. vs Super Cassettes Industries Lt...,1,0,1,0,1,0,10. Protecting Copyrights and Other Intelle...,Yes
5,Privacy,69052901,Naresh Trehan vs Rakesh Kumar Gupta on 24 Nove...,2,0,2,0,0,0,8. Exemption from disclosure of information.--...,No
6,Privacy,80759470,Kamal Bhasin vs Radha Krishna Mathur & Anr. on...,2,0,2,0,0,0,12. We are in agreement with the CIC and the c...,No
7,Privacy,70139862,"R.K.Jain vs Union Of India & Anr on 16 April, ...",4,0,4,0,0,0,The appellant filed an application to Cent...,No
8,force majeure,1716913,"Century Textiles And Industries Ltd., ... vs M...",8,0,8,0,6,0,18. FAILURE TO SUPPLY: \n(a) (i) The Board sha...,Yes
9,force majeure,172129667,Jdl Lime Stone & Dolomite Mines vs State Of Od...,4,0,4,0,2,0,4. Failure on the part of the lessee to fulfil...,Yes



** Review the table above. Do not proceed to full processing until satisfied. **


In [6]:
df.to_csv("batch2.csv")

In [ ]:
# Cell 4: Copy Base Tables to Target

# Copy stored_results
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size FROM stored_results")
    stored_rows = src_cur.fetchall()

inserted_sr = 0
with target_conn.cursor() as tgt_cur:
    for row in stored_rows:
        tgt_cur.execute("""
            INSERT INTO stored_results (Doc_ID, Title, Doc_Text, Doc_Blockquotes, Doc_Size)
            VALUES (%s, %s, %s, %s, %s)
            ON CONFLICT (Doc_ID) DO NOTHING
        """, row)
        inserted_sr += tgt_cur.rowcount
    target_conn.commit()

print(f"stored_results: {len(stored_rows)} source rows, {inserted_sr} newly inserted into target")

# Copy search_queries
with source_conn.cursor() as src_cur:
    src_cur.execute("SELECT searchquery, dateandtime FROM search_queries")
    sq_rows = src_cur.fetchall()

with target_conn.cursor() as tgt_cur:
    for row in sq_rows:
        tgt_cur.execute("""
            INSERT INTO search_queries (searchquery, dateandtime)
            VALUES (%s, %s)
        """, row)
    target_conn.commit()

print(f"search_queries: {len(sq_rows)} rows copied to target")

In [ ]:
# Cell 5: Load Checkpoint

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
    processed_ids = set(checkpoint.get('processed_ids', []))
    stats = checkpoint.get('stats', {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_errors': 0,
    })
    print(f"Checkpoint loaded: {len(processed_ids)} rows already processed")
else:
    processed_ids = set()
    stats = {
        'total_processed': 0,
        'total_expanded': 0,
        'total_classified_as_clause': 0,
        'total_errors': 0,
    }
    print("No checkpoint found. Starting fresh.")

In [ ]:
# Cell 6: Fetch & Prepare Work Queue

# Fetch all classified_index rows from source
with source_conn.cursor() as cur:
    cur.execute("""
        SELECT doc_id, title, searchquery, matching_indents, matching_columns,
               matching_columns_after_classification, matching_indents_after_classification
        FROM classified_index
    """)
    all_rows = cur.fetchall()

print(f"Total classified_index rows in source: {len(all_rows)}")

# Prefetch all Doc_Text from stored_results
doc_texts = {}
with source_conn.cursor() as cur:
    cur.execute("SELECT Doc_ID, Doc_Text FROM stored_results")
    for row in cur.fetchall():
        if row[1]:
            doc_texts[row[0]] = row[1]

print(f"Prefetched Doc_Text for {len(doc_texts)} documents")

# Build work queue, filtering out already-processed rows
work_queue = []
for row in all_rows:
    doc_id, title, searchquery, mi_raw, mc_raw, mc_after_raw, mi_after_raw = row
    composite_key = f"{doc_id}__{searchquery}"
    if composite_key in processed_ids:
        continue
    work_queue.append({
        'doc_id': doc_id,
        'title': title,
        'searchquery': searchquery,
        'matching_indents': insert_data._safe_eval(mi_raw),
        'matching_columns': insert_data._safe_eval(mc_raw),
        'matching_columns_after_classification': insert_data._safe_eval(mc_after_raw),
        'matching_indents_after_classification': insert_data._safe_eval(mi_after_raw),
        'composite_key': composite_key,
    })

print(f"{len(work_queue)} rows to process ({len(processed_ids)} already checkpointed)")

In [ ]:
# Cell 7: Process — Expand & Classify (main loop)
# Uses expand_and_classify_with_azure: one LLM call per snippet
# that determines clause boundaries AND classifies in a single pass.

from pipelineoperation import expand_and_classify_with_azure

def save_checkpoint():
    """Save current progress to checkpoint file."""
    checkpoint_data = {
        'processed_ids': list(processed_ids),
        'last_updated': datetime.now().isoformat(),
        'stats': stats,
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint_data, f, indent=2)


def expand_classify_with_retry(doc_text, snippet, max_retries=3):
    """Combined expand+classify with retry on transient failures."""
    for attempt in range(max_retries):
        try:
            return expand_and_classify_with_azure(doc_text, snippet)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise


insert_sql = """
    INSERT INTO classified_index(
        Doc_Id, Title, searchquery,
        matching_indents, matching_columns,
        matching_columns_after_classification,
        matching_indents_after_classification,
        expanded_columns, expanded_indents,
        expanded_columns_after_classification,
        expanded_indents_after_classification
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

total_batches = (len(work_queue) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Starting processing: {len(work_queue)} rows in {total_batches} batches")

for batch_idx in range(0, len(work_queue), BATCH_SIZE):
    batch = work_queue[batch_idx:batch_idx + BATCH_SIZE]
    batch_num = batch_idx // BATCH_SIZE + 1

    for row in batch:
        try:
            doc_text = doc_texts.get(row['doc_id'], '')

            # Combined expand + classify for matching_columns snippets
            mc_results = []
            for s in row['matching_columns']:
                mc_results.append(expand_classify_with_retry(doc_text, s))
                time.sleep(0.1)

            # Combined expand + classify for matching_indents snippets
            mi_results = []
            for s in row['matching_indents']:
                mi_results.append(expand_classify_with_retry(doc_text, s))
                time.sleep(0.1)

            # Derive all column values from the combined results
            expanded_columns = [r['clause_text'] for r in mc_results]
            expanded_indents = [r['clause_text'] for r in mi_results]
            expanded_columns_after = [r['clause_text'] for r in mc_results if r['is_contract_clause']]
            expanded_indents_after = [r['clause_text'] for r in mi_results if r['is_contract_clause']]

            # Original snippet classification derived from the same LLM judgment
            mc_after = [s for s, r in zip(row['matching_columns'], mc_results) if r['is_contract_clause']]
            mi_after = [s for s, r in zip(row['matching_indents'], mi_results) if r['is_contract_clause']]

            # Insert into target DB
            with target_conn.cursor() as tgt_cur:
                tgt_cur.execute(insert_sql, (
                    row['doc_id'],
                    row['title'],
                    row['searchquery'],
                    str(row['matching_indents']),
                    str(row['matching_columns']),
                    str(mc_after),
                    str(mi_after),
                    str(expanded_columns),
                    str(expanded_indents),
                    str(expanded_columns_after),
                    str(expanded_indents_after),
                ))

            # Track stats
            stats['total_expanded'] += len(expanded_columns) + len(expanded_indents)
            stats['total_classified_as_clause'] += (
                len(expanded_columns_after) + len(expanded_indents_after)
            )
            processed_ids.add(row['composite_key'])
            stats['total_processed'] += 1

        except Exception as e:
            print(f"  ERROR processing {row['composite_key']}: {e}")
            stats['total_errors'] += 1
            processed_ids.add(row['composite_key'])  # skip on future runs

    # Commit and checkpoint after each batch
    target_conn.commit()
    save_checkpoint()
    print(f"Batch {batch_num}/{total_batches} complete "
          f"({stats['total_processed']} total processed)")

In [ ]:
# Cell 8: Save Final Checkpoint & Summary

save_checkpoint()
print("Final checkpoint saved.\n")

print("=== Processing Summary ===")
print(f"  Total processed:             {stats['total_processed']}")
print(f"  Total expanded clauses:      {stats['total_expanded']}")
print(f"  Total classified as clause:  {stats['total_classified_as_clause']}")
print(f"  Total errors/skipped:        {stats['total_errors']}")

# Show sample results from target DB
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT doc_id, title, searchquery,
               expanded_columns, expanded_columns_after_classification
        FROM classified_index
        WHERE LENGTH(expanded_columns) > 4
        LIMIT 5
    """)
    sample_rows = cur.fetchall()

if sample_rows:
    print("\n=== Sample Results from Target DB ===")
    for r in sample_rows:
        print(f"  DocID: {r[0]}, Query: {r[2]}")
        print(f"    Title: {(r[1] or '')[:60]}")
        print(f"    Expanded cols: {(r[3] or '')[:100]}...")
        print(f"    Classified cols: {(r[4] or '')[:100]}...")
        print()

In [ ]:
# Cell 9: Verification

# Row counts in target
with target_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    target_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    target_sr_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM search_queries")
    target_sq_count = cur.fetchone()[0]

# Row counts in source
with source_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM classified_index")
    source_ci_count = cur.fetchone()[0]
    cur.execute("SELECT COUNT(*) FROM stored_results")
    source_sr_count = cur.fetchone()[0]

print("=== Row Count Comparison ===")
print(f"  classified_index:  source={source_ci_count}  target={target_ci_count}")
print(f"  stored_results:    source={source_sr_count}  target={target_sr_count}")
print(f"  search_queries:    target={target_sq_count}")

# Show sample rows with expanded + classified data
with target_conn.cursor() as cur:
    cur.execute("""
        SELECT doc_id, searchquery,
               expanded_columns, expanded_indents,
               expanded_columns_after_classification,
               expanded_indents_after_classification
        FROM classified_index
        WHERE LENGTH(expanded_columns_after_classification) > 4
           OR LENGTH(expanded_indents_after_classification) > 4
        LIMIT 5
    """)
    verification_rows = cur.fetchall()

if verification_rows:
    print("\n=== Sample Rows with Expanded + Classified Data ===")
    for r in verification_rows:
        print(f"  DocID: {r[0]}, Query: {r[1]}")
        print(f"    expanded_columns: {(r[2] or '')[:80]}...")
        print(f"    expanded_indents: {(r[3] or '')[:80]}...")
        print(f"    expanded_cols_after_class: {(r[4] or '')[:80]}...")
        print(f"    expanded_indents_after_class: {(r[5] or '')[:80]}...")
        print()
else:
    print("\nNo rows with non-empty classified expanded data found.")

# Close connections
source_conn.close()
target_conn.close()
print("\nBoth database connections closed.")